In [ ]:
!pip install newsapi-python

In [ ]:
!pip install feedparser

In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import gradio as gr
from sklearn.linear_model import PassiveAggressiveClassifier


In [ ]:
from newsapi import NewsApiClient

In [ ]:
import os

# Reads the NewsAPI key from a Colab secret / environment variable first.
# Falls back to the key already in this notebook so the project keeps
# working out of the box, but for anything beyond a class demo the key
# should live in Colab's Secrets panel instead of in the notebook.
NEWSAPI_KEY = os.environ.get("NEWSAPI_KEY", "c1d9338b793842bb9703e130ee7f1be7")
newsapi = NewsApiClient(api_key=NEWSAPI_KEY)


In [ ]:
def verify_with_news_sources(query):
    """
    Fallback verification when Google News RSS returns nothing.
    Queries NewsAPI directly and flags whether any well-known outlet
    is reporting the same story.
    """

    try:
        response = newsapi.get_everything(
            q=query,
            language="en",
            sort_by="relevancy",
            page_size=5
        )
    except Exception:
        return "News verification failed."

    articles = (response or {}).get("articles") or []

    if not articles:
        return "\u26a0\ufe0f No news articles found."

    titles = [a.get("title") for a in articles if a.get("title")]
    sources = [
        a["source"]["name"]
        for a in articles
        if a.get("source") and a["source"].get("name")
    ]

    trusted_sources = ["BBC News", "Reuters", "CNN", "The Guardian", "NDTV"]
    trusted_found = [s for s in sources if s in trusted_sources]

    result = ""

    if trusted_found:
        result += "\u2705 Verified by trusted news sources:\n"
        for s in trusted_found[:3]:
            result += "\u2022 " + s + "\n"
    else:
        result += "\u26a0\ufe0f Not verified by major news sources.\n"

    result += "\nRelated Headlines:\n"
    for t in titles[:5]:
        result += "\u2022 " + t + "\n"

    return result


In [ ]:
import feedparser
import urllib.parse
import socket
import re


def google_news_check(query, timeout=6):
    """
    Pulls the top headlines for `query` from the Google News RSS feed.
    Returns a deduplicated list of up to 5 {"source", "title"} dicts,
    or an empty list if nothing is found or the request fails - callers
    already treat an empty list as "fall back to NewsAPI".
    """

    query = re.sub(r"\s+", " ", query or "").strip()

    if not query:
        return []

    # Long queries return noisy/irrelevant RSS results, so keep only
    # the first 10 words regardless of what the caller passed in.
    words = query.split()
    if len(words) > 10:
        query = " ".join(words[:10])

    encoded_query = urllib.parse.quote(query)

    url = (
        f"https://news.google.com/rss/search?"
        f"q={encoded_query}"
        f"&hl=en-IN"
        f"&gl=IN"
        f"&ceid=IN:en"
    )

    # feedparser has no built-in timeout, so bound it with the socket
    # default - otherwise a slow/dead connection can hang the whole
    # Gradio callback instead of failing gracefully.
    previous_timeout = socket.getdefaulttimeout()
    try:
        socket.setdefaulttimeout(timeout)
        feed = feedparser.parse(url)
    except Exception:
        return []
    finally:
        socket.setdefaulttimeout(previous_timeout)

    if not getattr(feed, "entries", None):
        return []

    headlines = []
    seen = set()

    for entry in feed.entries:
        title = getattr(entry, "title", "") or ""
        title = title.strip()

        if not title:
            continue

        # Titles arrive as "Headline - Source" (e.g. "... - NDTV").
        # Capture the source name before stripping it off the title
        # so it can be displayed alongside the headline again.
        source_match = re.search(r"\s*-\s*([^-]+)$", title)
        if source_match:
            source_name = source_match.group(1).strip()
            title = title[:source_match.start()].strip()
        else:
            source_entry = getattr(entry, "source", None)
            source_name = getattr(source_entry, "title", None) if source_entry else None
            source_name = (source_name or "Unknown Source").strip()

        key = title.lower()
        if key not in seen:
            seen.add(key)
            headlines.append({"source": source_name, "title": title})

        if len(headlines) == 5:
            break

    return headlines


In [ ]:
try:
    df_fake = pd.read_csv("Fake.csv")
    df_true = pd.read_csv("True.csv")
except FileNotFoundError as err:
    raise FileNotFoundError(
        "Fake.csv / True.csv not found. Upload both files to the Colab "
        "session (Files panel on the left) and re-run this cell."
    ) from err

df_fake["label"] = 1
df_true["label"] = 0

df = pd.concat([df_fake, df_true], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df = df[["text", "label"]]
df.head()


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text"] = df["text"].apply(clean_text)


In [ ]:
X = df["text"]
y = df["label"]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    min_df=2
)

X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42
)

model = PassiveAggressiveClassifier(max_iter=1000)
model.fit(X_train, y_train)
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)


In [ ]:
from sentence_transformers import SentenceTransformer, util
import re

# Loaded once and reused everywhere - semantic_news_check, category
# detection, and the Gradio callback all share this single instance.
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")


def extract_lead_sentence(text, min_words=5, fallback_words=20):
    """
    Pulls out the most headline-like chunk of a longer article.

    Sentence embeddings compare much more reliably when both sides are
    similar in length and style, so every semantic comparison in this
    notebook (news-headline matching and category detection) works off
    this same short extract instead of the raw article - see the note
    in detect_news_category() below for why that matters.
    """
    text = (text or "").strip()
    if not text:
        return ""

    first_sentence = re.split(r'[.!?\n]', text)[0]

    if len(first_sentence.split()) < min_words:
        first_sentence = " ".join(text.split()[:fallback_words])

    return re.sub(r"\s+", " ", first_sentence).strip()


def semantic_news_check(user_text, headlines):

    if not headlines:
        return 0.0, "\u26a0\ufe0f No Google News headlines available for semantic comparison."

    lead_text = extract_lead_sentence(user_text)

    user_embedding = semantic_model.encode(
        lead_text,
        convert_to_tensor=True
    )

    headline_embeddings = semantic_model.encode(
        headlines,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        user_embedding,
        headline_embeddings
    )[0]

    best_index = int(similarities.argmax())
    best_score = float(similarities[best_index])
    matched_headline = headlines[best_index]

    # Small bonus when the matched headline shares several exact
    # words with the article (names, places, numbers) - catches cases
    # where phrasing differs but the underlying story is clearly the
    # same one.
    user_words = set(lead_text.lower().split())
    headline_words = set(matched_headline.lower().split())
    common_words = user_words.intersection(headline_words)

    if len(common_words) >= 3:
        best_score += 0.05

    best_score = min(best_score, 1.0)

    result = f"""
\U0001f9e0 Highest Semantic Similarity : {round(best_score*100,2)}%

\U0001f4f0 Best Matched Headline

{matched_headline}
"""

    if best_score >= 0.75:
        result += """

\u2705 High semantic similarity.
Article strongly matches trusted news.
"""
    elif best_score >= 0.50:
        result += """

\u26a0\ufe0f Moderate semantic similarity.
Manual verification is recommended.
"""
    else:
        result += """

\u274c Low semantic similarity.
This article could be misleading or modified.
"""

    return best_score, result


In [ ]:
from sentence_transformers import util

# -----------------------------------------------------------------
# Semantic category detection - no keyword matching
# -----------------------------------------------------------------
# ROOT CAUSE of the old "always General" bug:
# the previous version embedded the *whole* article and compared it
# against ONE long, multi-topic description per category (e.g. a full
# sentence listing "cricket, football, tennis, badminton, Olympics...").
# Sentence embeddings are sensitive to length/style mismatch - a short
# headline compared against a long descriptive paragraph scores low
# even when the topic is a clean match, because the paragraph's
# embedding is diluted across many different words. That's why a
# clearly sporty headline ("Wimbledon debut, favourite tennis star")
# was still landing under the similarity floor.
#
# FIX: represent every category with several short, headline-style
# prototype phrases instead of one long paragraph, embed the article's
# lead sentence (same short extract semantic_news_check uses) instead
# of the full text, and score each category by its BEST-matching
# prototype (max-pooling). Short-to-short comparisons are what SBERT
# models are actually tuned for, so scores come out far more reliable.
# -----------------------------------------------------------------

CATEGORY_PROTOTYPES = {
    "Sports News": [
        "a cricket match result and highlights",
        "a football team wins a tournament",
        "a tennis player competes at a Grand Slam",
        "an Olympic athlete breaks a record",
        "sports league standings and player transfers",
    ],
    "Technology News": [
        "a new smartphone launch and specifications",
        "an artificial intelligence breakthrough",
        "a tech company releases a software update",
        "a cybersecurity data breach",
        "a startup raises funding",
    ],
    "Business News": [
        "the stock market rises or falls",
        "a company reports quarterly earnings",
        "RBI announces an interest rate decision",
        "a company lists its IPO on the stock exchange",
        "economy inflation and GDP growth figures",
    ],
    "Entertainment News": [
        "an actor's new movie releases in theatres",
        "a celebrity wedding or gossip story",
        "a music album or concert tour announcement",
        "an OTT platform announces a new web series",
        "a television show wins an award",
    ],
    "National News": [
        "the Indian parliament passes a new bill",
        "the prime minister announces a policy",
        "state government election results are declared",
        "the Supreme Court delivers a verdict",
        "the central government launches a welfare scheme",
    ],
    "International News": [
        "a war breaks out between two countries",
        "the United Nations discusses a global conflict",
        "two countries hold diplomatic talks",
        "world leaders meet at an international summit",
        "countries sign an international trade agreement",
    ],
    "Health News": [
        "a hospital treats a disease outbreak",
        "vaccine trial results are announced",
        "doctors warn about a health risk",
        "a new medicine is approved for treatment",
        "a public health awareness campaign launches",
    ],
    "Crime News": [
        "police arrest a suspect in a case",
        "a court sentences the accused in a trial",
        "investigators probe a cybercrime or fraud case",
        "a murder case investigation update",
        "law enforcement cracks down on organised crime",
    ],
    "Education News": [
        "a university announces its admission process",
        "school board exam results are declared",
        "the government announces a new education policy",
        "a college entrance exam schedule is released",
        "a scholarship programme is launched for students",
    ],
}

# Keyword mapping used as a first pass before the semantic check -
# catches unambiguous cases quickly and stops them from being
# diluted into the wrong category by the embedding comparison.
CATEGORY_KEYWORDS = {
    "Sports News": [
        "cricket", "football", "tennis", "hockey", "olympics", "match",
        "tournament", "ipl", "fifa", "wimbledon", "medal", "athlete",
        "goal", "wicket", "kabaddi", "badminton",
    ],
    "Technology News": [
        "smartphone", "ai", "artificial intelligence", "software", "app",
        "cybersecurity", "startup", "chip", "processor", "gadget",
        "app update", "data breach", "robot", "software update",
    ],
    "Business News": [
        "stock market", "shares", "ipo", "rbi", "inflation", "gdp",
        "earnings", "sensex", "nifty", "economy", "merger", "investment",
        "interest rate", "quarterly results",
    ],
    "Entertainment News": [
        "movie", "film", "actor", "actress", "celebrity", "box office",
        "music album", "concert", "ott", "web series", "bollywood",
        "hollywood", "award show",
    ],
    "National News": [
        "parliament", "prime minister", "lok sabha", "rajya sabha",
        "state government", "election results", "supreme court",
        "welfare scheme", "chief minister", "cabinet",
    ],
    "International News": [
        "united nations", "war", "ceasefire", "diplomatic", "summit",
        "foreign minister", "trade agreement", "un security council",
        "invasion", "border conflict",
    ],
    "Health News": [
        "hospital", "vaccine", "outbreak", "disease", "health ministry",
        "doctors", "medicine", "who", "epidemic", "treatment", "virus",
    ],
    "Crime News": [
        "police", "arrest", "murder", "court", "sentenced", "investigation",
        "fraud", "cybercrime", "fir", "accused", "crime branch",
    ],
    "Education News": [
        "university", "admission", "board exam", "education policy",
        "entrance exam", "scholarship", "school", "college", "neet", "jee",
    ],
}

_PROTOTYPE_CATEGORIES = []
_PROTOTYPE_TEXTS = []

for _category, _phrases in CATEGORY_PROTOTYPES.items():
    for _phrase in _phrases:
        _PROTOTYPE_CATEGORIES.append(_category)
        _PROTOTYPE_TEXTS.append(_phrase)

# Computed once at load time and reused for every prediction - this is
# the only place these 45 short phrases get encoded.
_PROTOTYPE_EMBEDDINGS = semantic_model.encode(
    _PROTOTYPE_TEXTS,
    convert_to_tensor=True
)

# Prints the per-category score on every call so a mismatch can be
# debugged during development. Leave this off for the live demo -
# flip it back to True if a category ever looks wrong again.
DEBUG_CATEGORY_DETECTION = False


def detect_news_category(text, min_similarity=0.32):
    """
    Detects the most likely news category using a keyword pass first,
    then semantic similarity against short, headline-style prototype
    phrases (see comment above) as the fallback. Always returns one
    of the 9 exact categories - "General" is never returned; the
    semantic best-match is used even on a low-confidence score.
    """

    lead_text = extract_lead_sentence(text)

    if not lead_text:
        return "Unknown"

    # Keyword pass - unambiguous, fast, and avoids embedding dilution
    # on short headlines that contain a clear category-defining word.
    lower_lead_text = lead_text.lower()
    keyword_hits = {}
    for category_name, keywords in CATEGORY_KEYWORDS.items():
        hits = sum(1 for kw in keywords if kw in lower_lead_text)
        if hits:
            keyword_hits[category_name] = hits

    if keyword_hits:
        detected = max(keyword_hits, key=keyword_hits.get)

        if DEBUG_CATEGORY_DETECTION:
            print("----- Keyword hits -----")
            for name, hits in sorted(keyword_hits.items(), key=lambda kv: -kv[1]):
                print(f"{name} : {hits}")
            print(f"Detected: {detected} (via keyword match)")
            print("---------------------------------------")

        return detected

    # Semantic fallback - no keyword matched, so fall back to the
    # best-scoring prototype. Always returns an exact category.
    text_embedding = semantic_model.encode(lead_text, convert_to_tensor=True)
    similarities = util.cos_sim(text_embedding, _PROTOTYPE_EMBEDDINGS)[0].tolist()

    category_scores = {}
    for category_name, score in zip(_PROTOTYPE_CATEGORIES, similarities):
        if score > category_scores.get(category_name, -1.0):
            category_scores[category_name] = score

    best_category = max(category_scores, key=category_scores.get)
    best_score = category_scores[best_category]

    detected = best_category

    if DEBUG_CATEGORY_DETECTION:
        print("----- Category similarity scores -----")
        for name, score in sorted(category_scores.items(), key=lambda kv: -kv[1]):
            print(f"{name} : {round(score, 3)}")
        print(f"Detected: {detected} (best score = {round(best_score, 3)})")
        print("---------------------------------------")

    return detected


In [ ]:
import numpy as np


def predict_news(category, text, lang):

    if text.strip() == "":
        return (
            "",
            """
            <div style="
            background:#2b2f38;
            color:#e2c56a;
            padding:15px;
            border-radius:12px;
            border:1px solid #e2c56a;
            ">
            \u26a0\ufe0f Please paste a news headline or article before verifying.
            </div>
            """,
        )

    # -----------------------------
    # Category Detection (semantic) + Validation
    # -----------------------------
    # Detection is only for user guidance - it never changes the
    # fake/real prediction below.
    detected_category = detect_news_category(text)

    warning_message = ""

    if (
        detected_category not in ["Unknown", "General"]
        and detected_category != category
    ):
        warning_message = f"""
        <div style="
        background:#3a2f12;
        color:#f0c96b;
        padding:15px;
        border-radius:12px;
        margin-bottom:15px;
        border:1px solid #6b5620;
        ">
        \u26a0 <b>Category Mismatch</b><br>
        Selected: <b>{category}</b> &nbsp;|&nbsp; Detected: <b>{detected_category}</b>
        <br><br>
        The prediction below is unaffected by category - it's generated purely from the article text.
        </div>
        """

    # -----------------------------
    # ML Fake/Real Prediction (unaffected by category)
    # -----------------------------
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    score = model.decision_function(vec)[0]

    fake_prob = round((1 / (1 + np.exp(-score))) * 100, 1)
    real_prob = round(100 - fake_prob, 1)

    confidence = max(real_prob, fake_prob)

    # -----------------------------
    # Google News + NewsAPI Verification
    # -----------------------------
    search_query = extract_lead_sentence(text)
    headlines_data = google_news_check(search_query)
    headlines = [h["title"] for h in headlines_data]

    if headlines_data:
        verification = (
            f"\u2705 Found {len(headlines_data)} related articles from trusted Google News sources."
        )
        headlines_text = "<br>".join(
            [f"\u2022 {h['source']}<br>{h['title']}" for h in headlines_data]
        )
    else:
        verification = verify_with_news_sources(search_query)
        headlines_text = "\u26a0\ufe0f No Google News headlines found."

    # gr.HTML() can't render "\n" as a line break, so convert it for
    # correct display.
    verification_html = verification.replace("\n", "<br>")

    # -----------------------------
    # Semantic Similarity Check
    # -----------------------------
    semantic_score, semantic_result = semantic_news_check(text, headlines)
    semantic_result_html = semantic_result.replace("\n", "<br>")

    # -----------------------------
    # Analysis Summary
    # -----------------------------
    model_prediction = "REAL" if real_prob > fake_prob else "FAKE"

    if headlines:
        news_match_summary = f"Found {len(headlines)} similar headlines on Google News"
    else:
        news_match_summary = "No matching Google News headlines found"

    summary_rows = [
        ("Selected Category", category),
        ("Detected Category", detected_category),
        ("Model Confidence", f"{confidence}%"),
        ("Google News Match", news_match_summary),
    ]

    summary_html = "".join(
        f"""<div style="display:flex;justify-content:space-between;
        padding:6px 0;border-bottom:1px solid rgba(255,255,255,0.08);">
        <span style="opacity:0.75;">{label}</span><b>{value}</b></div>"""
        for label, value in summary_rows
    )

    # ===============================
    # SMART VERDICT ENGINE V2
    # ===============================
    headline_count = len(headlines)

    if headline_count >= 3 and semantic_score >= 0.50:
        verdict = "\u2705 LIKELY REAL NEWS"
        credibility = min(99, int((semantic_score * 75) + (headline_count * 7)))

    elif headline_count >= 1 and semantic_score >= 0.35:
        verdict = "\u26a0\ufe0f PROBABLY REAL - NEEDS FACT CHECKING"
        credibility = min(90, int((semantic_score * 70) + (headline_count * 6)))

    elif headline_count >= 3:
        verdict = "\u26a0\ufe0f RELATED NEWS FOUND BUT CONTENT DIFFERS"
        credibility = 55

    else:
        verdict = "\u2757 POSSIBLY FAKE OR UNVERIFIED"
        credibility = max(20, int(semantic_score * 100))

    if "LIKELY REAL" in verdict:
        verdict_color, glow = "#28a745", "#00ff88"
    elif "PROBABLY REAL" in verdict:
        verdict_color, glow = "#ffc107", "#ffdd57"
    else:
        verdict_color, glow = "#dc3545", "#ff4d6d"

    verdict_card = f"""
    <div style="
    background:{verdict_color};
    padding:20px;
    border-radius:20px;
    text-align:center;
    font-size:24px;
    font-weight:bold;
    color:white;
    margin-top:15px;
    box-shadow:0px 0px 25px {glow};
    ">
    \u2b50 FINAL VERDICT \u2b50
    <br><br>
    {verdict}
    <br><br>
    \U0001f3af Credibility Score: {credibility}/100
    </div>
    """

    full_explanation = f"""
    {warning_message}
    <h3>\U0001f50d Analysis Summary</h3>
    {summary_html}
    <hr>
    <h3>\U0001f9e0 Semantic Similarity Check</h3>
    {semantic_result_html}
    <hr>
    <h3>\U0001f50e News Verification</h3>
    {verification_html}
    <hr>
    <h3>\U0001f4f0 Google News Headlines</h3>
    {headlines_text}
    """

    return verdict_card, full_explanation


In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# \U0001f4f0 Fake News Detection System")
    gr.Markdown("### AI-Powered Fake News Verification using Machine Learning + Google News + Semantic Analysis")

    with gr.Row():
        with gr.Column(scale=5):

            category = gr.Radio(
                choices=[
                    "National News",
                    "International News",
                    "Sports News",
                    "Entertainment News",
                    "Business News",
                    "Technology News",
                    "Health News",
                    "Crime News",
                    "Education News"
                ],
                label="Select News Category",
                value="National News"
            )

            text_input = gr.Textbox(
                label="Paste News Article",
                lines=6,
                placeholder="Paste any news headline or article here..."
            )

            lang = gr.Dropdown(
                ["English"],
                value="English",
                label="Language"
            )

            with gr.Row():
                submit = gr.Button("\U0001f680 Verify News", variant="primary")
                clear = gr.Button("\U0001f5d1\ufe0f Clear")

        with gr.Column(scale=6):
            verdict_output = gr.HTML()
            explanation_output = gr.HTML()

    submit.click(
        predict_news,
        [category, text_input, lang],
        [verdict_output, explanation_output]
    )

    clear.click(
        lambda: ("", ""),
        outputs=[verdict_output, explanation_output]
    )

demo.launch(share=True, debug=True)
